# 05 - Prepare the validation ablation

Configuration, dataset verification, verified backups, and dev-baseline reuse.

## Fixed configuration
619 dev abstracts, penalties 0.5/1.0/2.0, ten beams, five groups, five candidates, and max_length=64.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT / 'configs/config.yaml').exists(), 'Open from the project root or notebooks directory'
assert Path(sys.prefix).resolve() == (ROOT / '.venv-scoring').resolve(), 'Select the .venv-scoring kernel and restart'
PHASE_NOTEBOOKS = ['05_prepare_validation_ablation.ipynb', '06_generate_dbs_candidates.ipynb', '07_alignscore_reranking.ipynb', '08_evaluate_summac_rouge.ipynb', '09_select_diversity_penalty.ipynb']
CONFIG = {'seed': 42,
 'dataset': 'data/raw/dev.jsonl',
 'split': 'dev',
 'expected_samples': 619,
 'checkpoint': 'outputs/hub_models/SciTLDR-trained-BART',
 'checkpoint_source': 'SamHuertas/SciTLDR-trained-BART',
 'checkpoint_revision': '36ec27a863a66fdce31240f6d58fd6da2aa1b28d',
 'output': 'outputs/validation',
 'source_max_length': 512,
 'batch_size': 4,
 'device': 'cuda',
 'penalties': [0.5, 1.0, 2.0],
 'generation': {'num_beams': 10,
                'num_beam_groups': 5,
                'num_return_sequences': 5,
                'do_sample': False,
                'max_length': 64,
                'max_new_tokens': None},
 'baseline': {'num_beams': 1,
              'num_beam_groups': 1,
              'num_return_sequences': 1,
              'diversity_penalty': 0.0,
              'do_sample': False,
              'max_new_tokens': 64,
              'no_repeat_ngram_size': 3},
 'alignscore': {'model': 'roberta-base',
                'evaluation_mode': 'nli_sp',
                'batch_size': 8,
                'checkpoint': 'checkpoints/AlignScore-base.ckpt'},
 'summac': {'model_name': 'vitc',
            'granularity': 'sentence',
            'op1': 'max',
            'op2': 'mean',
            'use_ent': True,
            'use_con': False,
            'max_doc_sents': 100},
 'rouge': {'metric': 'rougeL', 'use_stemmer': True, 'references': 'max'},
 'metric_snapshots': {'roberta-base': 'e2da8e2f811d1448a5b465c236feacd80ffbac7b',
                      'tals/albert-xlarge-vitaminc-mnli': '3082ba54344bd9ddada2be1c5e9b4131721d2a5d'},
 'baseline_source': 'outputs/backup/pre_dbs_ablation_20261111_181024_667197/validation/val_baseline.csv',
 'dbs_implementation': {'repository': 'transformers-community/group-beam-search',
                        'revision': '56d0e48dadff53cb8b8653981b10534c50707fb1',
                        'compatibility': 'Remove v5-only next_sequence_length/is_first_iteration '
                                         'kwargs for Transformers 4.57 BART; algorithm unchanged.'}}


## Shared utilities


In [2]:
"""Shared validation, backup, and resume definitions; no model inference."""
import ast
import csv
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import shutil
import time
from datetime import datetime
from zoneinfo import ZoneInfo




def require(ok, message):
    if not ok:
        raise ValueError(message)


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    replace_file(tmp, path)


def replace_file(source, destination):
    # Windows indexers/antivirus can briefly hold a just-written result file.
    for attempt in range(20):
        try:
            os.replace(source, destination)
            return
        except PermissionError:
            if attempt == 19:
                raise
            time.sleep(0.1)


def save_csv(path, rows):
    require(bool(rows), f'Cannot save empty table: {path}')
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.csv.tmp')
    with tmp.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    replace_file(tmp, path)


def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def versions():
    result = {}
    for name in ['torch', 'transformers', 'summac', 'alignscore', 'rouge-score', 'pytorch-lightning', 'spacy', 'nltk']:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result



def notebook_code_sha256():
    # Compare Python syntax, ignoring comments, formatting, and empty code cells.
    hashes = {}
    for name in PHASE_NOTEBOOKS:
        notebook = ROOT / 'notebooks' / name
        nb = json.loads(notebook.read_text(encoding='utf-8'))
        code = []
        for index, cell in enumerate(nb['cells']):
            if cell['cell_type'] != 'code':
                continue
            tree = ast.parse(''.join(cell['source']), filename=f'{name}:cell{index}')
            if tree.body:
                code.append(ast.dump(tree, include_attributes=False))
        hashes[name] = hashlib.sha256(json.dumps(code, ensure_ascii=False).encode('utf-8')).hexdigest()
    return hashes


def identity_differences(saved, current, prefix=''):
    differences = []
    if isinstance(saved, dict) and isinstance(current, dict):
        for key in sorted(set(saved) | set(current)):
            field = f'{prefix}.{key}' if prefix else key
            if key not in saved or key not in current:
                differences.append(field)
            else:
                differences.extend(identity_differences(saved[key], current[key], field))
    elif saved != current:
        differences.append(prefix)
    return differences


In [3]:
class SummarizationAblation:
    def __init__(self, config):
        self.c = json.loads(json.dumps(config))
        c = self.c
        require(c['split'] == 'dev', 'Only the validation dev split is permitted')
        require(c['penalties'] == [0.5, 1.0, 2.0], 'Expected diversity penalties: 0.5, 1.0, 2.0')
        require(c['generation'] == dict(num_beams=10, num_beam_groups=5, num_return_sequences=5,
            do_sample=False, max_length=64, max_new_tokens=None), 'Expected 10 beams, 5 groups, 5 candidates, deterministic decoding, and max_length=64')
        self.out = (ROOT / c['output']).resolve()
        require(self.out == (ROOT / 'outputs/validation').resolve(), 'Use outputs/validation')
        self.data = []
        for line in (ROOT / c['dataset']).read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            r = json.loads(line)
            self.data.append(dict(id=r['paper_id'], abstract=' '.join(' '.join(r['source']).split()),
                                  human_tldrs=[' '.join(x.split()) for x in r['target']]))
        require(len(self.data) == c['expected_samples'] == 619,
                f'Validation count is {len(self.data)}, expected 619. Investigate before proceeding.')
        require(len({r['id'] for r in self.data}) == len(self.data), 'Nonunique abstract IDs')
        for r in self.data:
            require(r['id'] and r['abstract'] and r['human_tldrs'] and all(r['human_tldrs']), 'Invalid validation record')
        self.ids = [r['id'] for r in self.data]


    def run_identity(self):
        model = ROOT / self.c['checkpoint']
        require((model / 'model.safetensors').is_file(), 'Missing BART checkpoint')
        files = {p.name: sha(p) for p in sorted(model.iterdir()) if p.is_file() and not p.name.startswith('.')}
        metric_code = {}
        for package, names in {'summac': ['model_summac.py'], 'alignscore': ['model.py', 'inference.py']}.items():
            dist = importlib.metadata.distribution(package)
            for name in names:
                metric_code[package + '/' + name] = sha(dist.locate_file(package + '/' + name))
        return dict(config=self.c, dataset_sha256=sha(ROOT / self.c['dataset']), metric_code=metric_code,
                    ordered_ids=self.ids, model_files=files, versions=versions(),
                    notebook_code_sha256=notebook_code_sha256(),
                    notebook_fingerprint_format="python-ast-v1",
                    alignscore_checkpoint_sha256=sha(ROOT / self.c['alignscore']['checkpoint']))


    def inspect_inputs(self):
        baseline = ROOT / 'outputs/baseline.csv'
        old = read_csv(baseline) if baseline.exists() else []
        print(json.dumps(dict(validation_count=len(self.data), validation_unique_ids=len(set(self.ids)),
            existing_baseline_rows=len(old), existing_baseline_validation_overlap=len(set(self.ids) & {r['id'] for r in old}),
            selected_model=self.c['checkpoint'], alignscore_checkpoint_exists=(ROOT / self.c['alignscore']['checkpoint']).is_file(),
            disk_free_bytes=shutil.disk_usage(ROOT).free, versions=versions()), indent=2))


    def has_preparation_only(self):
        # Never refresh a run containing any candidate, reranking, or evaluation output.
        allowed = {'experiment_manifest.json', 'backup_verification.json',
                   'val_baseline.csv', 'val_baseline.receipt.json',
                   'progress/reuse_baseline.json'}
        existing = {p.relative_to(self.out).as_posix() for p in self.out.rglob('*') if p.is_file()}
        return existing <= allowed


    def prepare_outputs(self):
        run_identity = self.run_identity()
        manifest = self.out / 'experiment_manifest.json'
        if manifest.exists():
            saved = json.loads(manifest.read_text(encoding='utf-8'))
            changed = identity_differences(saved, run_identity)
            if not changed:
                require((self.out / 'backup_verification.json').exists(), 'Missing backup verification')
                print('Existing compatible experiment; resume permitted')
                return
            code_fields = {'notebook_code_sha256', 'notebook_fingerprint_format'}
            code_only = all(field.split('.')[0] in code_fields for field in changed)
            require(code_only and self.has_preparation_only(),
                    'Preparation identity changed: ' + ', '.join(changed) +
                    '. See ABLATION_RUN_GUIDE.md recovery steps; existing results have not been modified.')
            baseline = self.out / 'val_baseline.csv'
            if baseline.exists():
                self.load_completed_output(baseline)
                require((ROOT / self.c['baseline_source']).is_file(),
                        'The reusable baseline source is missing; existing preparation was preserved.')
            print('Notebook definitions changed before candidate generation. Archiving preparation for a fresh identity.')
        else:
            require(not any(p.is_file() for p in self.out.rglob('*')),
                    'Conflicting validation files; see ABLATION_RUN_GUIDE.md recovery steps')
        self.backup_outputs()
        save_json(self.out / 'backup_verification.json', dict(conflicts_remaining=False,
            preserved_inputs=['outputs/baseline.csv', self.c['checkpoint'], self.c['dataset']],
            note='Legacy faithfulness/run_01 is a separate test experiment, left in place.'))
        save_json(manifest, run_identity)


    def backup_outputs(self):
        # Archive active validation outputs while preserving model and dataset inputs.
        files = sorted(p for p in self.out.rglob('*') if p.is_file()) if self.out.exists() else []
        stamp = datetime.now(ZoneInfo('Asia/Taipei')).strftime('%Y%m%d_%H%M%S_%f')
        target = ROOT / 'outputs/backup' / ('pre_dbs_ablation_' + stamp)
        require(target.resolve().is_relative_to((ROOT / 'outputs/backup').resolve()), 'Unsafe backup path')
        target.mkdir(parents=True, exist_ok=False)
        record = dict(status='in_progress', files=[])
        save_json(target / 'backup_manifest.json', record)
        for p in files:
            require(p.resolve().is_relative_to(self.out), 'Output symlinks are not permitted')
            dest = target / 'validation' / p.relative_to(self.out)
            entry = dict(original=str(p.relative_to(ROOT)), backup=str(dest.relative_to(ROOT)), sha256=sha(p))
            record['files'].append({**entry, 'status': 'planned'})
            save_json(target / 'backup_manifest.json', record)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(p), str(dest))
            require(not p.exists() and sha(dest) == entry['sha256'], f'Backup verification failed: {p}')
            record['files'][-1]['status'] = 'verified'
            save_json(target / 'backup_manifest.json', record)
        require(not any(p.is_file() for p in self.out.rglob('*')), 'Conflicts remain')
        record['status'] = 'verified'
        save_json(target / 'backup_manifest.json', record)
        print('Verified backup:', target)


    def validate_run(self):
        p = self.out / 'experiment_manifest.json'
        require(p.exists(), 'Preparation manifest is missing; see ABLATION_RUN_GUIDE.md')
        changed = identity_differences(json.loads(p.read_text(encoding='utf-8')), self.run_identity())
        code_fields = {'notebook_code_sha256', 'notebook_fingerprint_format'}
        code_only = changed and all(field.split('.')[0] in code_fields for field in changed)
        if code_only and self.has_preparation_only():
            had_baseline = (self.out / 'val_baseline.csv').is_file()
            self.prepare_outputs()
            if had_baseline:
                self.reuse_validation_baseline()
            changed = identity_differences(json.loads(p.read_text(encoding='utf-8')), self.run_identity())
        require(not changed,
                'Resume identity mismatch: ' + ', '.join(changed) +
                '. No results were modified. See ABLATION_RUN_GUIDE.md recovery steps.')


    def configure_metric_environment(self):
        # Require the inspected immutable snapshots, then prohibit network resolution.
        from huggingface_hub import snapshot_download
        for model, revision in self.c['metric_snapshots'].items():
            resolved = Path(snapshot_download(model, local_files_only=True))
            require(resolved.name == revision, f'Metric cache revision changed: {model}')
        os.environ['HF_HUB_OFFLINE'] = '1'
        import huggingface_hub.constants
        huggingface_hub.constants.HF_HUB_OFFLINE = True
        import transformers.utils.hub
        transformers.utils.hub._is_offline_mode = True


    def get_stage_manifest(self, name, inputs):
        self.validate_run()
        path = self.out / 'progress' / (name + '.json')
        value = dict(experiment_sha256=sha(self.out / 'experiment_manifest.json'),
                     inputs={str(p.relative_to(ROOT)): sha(p) for p in inputs})
        if path.exists():
            require(json.loads(path.read_text()) == value, f'Changed stage inputs: {name}')
        else:
            # Reject orphan outputs through each stage's output manifest guard.
            save_json(path, value)
        return path


    def load_progress(self, path, expected, stage_manifest):
        if not path.exists():
            return []
        receipt = path.with_suffix('.receipt.json')
        require(receipt.exists(), f'Untracked output: {path}')
        require(json.loads(receipt.read_text()) == dict(stage_sha256=sha(stage_manifest), output_sha256=sha(path)),
                f'Modified or incompatible output: {path}')
        rows = read_csv(path)
        require([r['id'] for r in rows] == [r['id'] for r in expected[:len(rows)]], f'Invalid saved ID order: {path}')
        require(len(rows) <= len(expected), 'Too many saved rows')
        return rows


    def save_progress(self, path, rows, stage_manifest):
        save_csv(path, rows)
        save_json(path.with_suffix('.receipt.json'), dict(stage_sha256=sha(stage_manifest), output_sha256=sha(path)))


    def load_completed_output(self, path):
        receipt = path.with_suffix('.receipt.json')
        require(path.exists() and receipt.exists(), f'Missing stage output: {path}')
        require(json.loads(receipt.read_text())['output_sha256'] == sha(path), f'Changed output: {path}')
        rows = read_csv(path)
        require([r['id'] for r in rows] == self.ids, f'Incomplete or misaligned output: {path}')
        return rows


    def reuse_validation_baseline(self):
        self.validate_run()
        source = ROOT / self.c['baseline_source']
        require(source.exists(), 'Reusable dev baseline is absent')
        prior_manifest = source.parent / 'experiment_manifest.json'
        require(prior_manifest.exists(), 'Reusable baseline has no provenance manifest')
        prior = json.loads(prior_manifest.read_text(encoding='utf-8'))
        require(prior['model_files'] == self.run_identity()['model_files'], 'Baseline checkpoint/tokenizer differs')
        require(prior['dataset_sha256'] == sha(ROOT / self.c['dataset']), 'Baseline source dataset differs')
        require(prior['config']['baseline'] == self.c['baseline'], 'Baseline decoding differs')
        require(prior['config']['source_max_length'] == self.c['source_max_length'], 'Baseline source limit differs')
        require(prior['config']['seed'] == self.c['seed'], 'Baseline seed differs')
        rows = self.load_completed_output(source)
        for row, expected in zip(rows, self.data):
            require(row['abstract'] == expected['abstract'], 'Baseline abstract mismatch')
            require(json.loads(row['human_tldrs']) == expected['human_tldrs'], 'Baseline reference mismatch')
            require(bool(row['summary'].strip()), 'Blank baseline summary')
        receipt = self.get_stage_manifest('reuse_baseline', [source, prior_manifest])
        dest = self.out / 'val_baseline.csv'
        existing = self.load_progress(dest, self.data, receipt)
        require(not existing or existing == rows, 'Active baseline differs from the verified baseline source')
        if not existing:
            self.save_progress(dest, rows, receipt)
        print('Verified and reused', len(rows), 'dev baseline summaries. Test baseline untouched.')





def valid_scores(values, count):
    require(len(values) == count and all(math.isfinite(x) and 0 <= x <= 1 for x in values), 'Invalid score count/range')



## Input audit


In [4]:
ablation = SummarizationAblation(CONFIG)
ablation.inspect_inputs()
require((ROOT / CONFIG['alignscore']['checkpoint']).is_file(), 'Missing AlignScore checkpoint; see the run guide')


{
  "validation_count": 619,
  "validation_unique_ids": 619,
  "existing_baseline_rows": 618,
  "existing_baseline_validation_overlap": 0,
  "selected_model": "outputs/hub_models/SciTLDR-trained-BART",
  "alignscore_checkpoint_exists": true,
  "disk_free_bytes": 86656880640,
  "versions": {
    "torch": "2.11.0+cu128",
    "transformers": "4.57.6",
    "summac": "0.0.4",
    "alignscore": "0.1.3",
    "rouge-score": "0.1.2",
    "pytorch-lightning": "1.9.5",
    "spacy": "3.8.18",
    "nltk": "3.10.4"
  }
}


## Verified output preparation



In [6]:
ablation.prepare_outputs()


Verified backup: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\outputs\backup\pre_dbs_ablation_20261111_185048_301857


## Validation baseline reuse
The preserved dev baseline is verified against model/tokenizer hashes, dataset hash, IDs, abstracts, references, source limit, seed, and greedy decoding settings before copying into the active run. 

In [7]:
baseline_source = ROOT / CONFIG['baseline_source']
if baseline_source.is_file():
    ablation.reuse_validation_baseline()
else:
    print('Reusable dev baseline source: absent')
print('Preparation manifest:', (ablation.out / 'experiment_manifest.json').is_file())
print('Active dev baseline:', (ablation.out / 'val_baseline.csv').is_file())


Verified and reused 619 dev baseline summaries. Test baseline untouched.
Preparation manifest: True
Active dev baseline: True
